This code snippet provides the core steps, including the necessary $\text{ICD-9}$ mapping function and the logic to implement the plan above.

In [ ]:
# --- Setup ---
import pandas as pd
import numpy as np

np.random.seed(42)

# --- Configuration ---
RAW_DATA_PATH = '../data/raw/diabetic_data.csv' 
PROCESSED_DATA_PATH = '../data/processed/patient_records_cleaned.csv'

# --- 1. Load Data and Initial Cleanup ---
# Read '?' as NaN during load
df = pd.read_csv(RAW_DATA_PATH, na_values=['?', 'Unknown/Invalid']) 
print(f"Initial shape: {df.shape}")

# Drop columns with high missing rates or non-analytic value
# 'weight' is often over 90% missing. 'encounter_id' and 'patient_nbr' are kept temporarily.
df = df.drop(columns=['weight', 'payer_code', 'medical_specialty']) 

# Rename columns to snake_case for consistency with the target schema
df = df.rename(columns={
    'patient_nbr': 'Patient_ID',
    'time_in_hospital': 'Length_of_Stay',
    'number_outpatient': 'Num_Outpatient',
    'number_emergency': 'Num_Emergency',
    'number_inpatient': 'Num_Inpatient',
    'num_lab_procedures': 'Num_Lab_Procedures',
    'num_procedures': 'Num_Procedures',
    'num_medications': 'Num_Medications',
    'number_diagnoses': 'Num_Diagnoses_Total'
})

In [ ]:
# --- 2. Filter Invalid Encounters ---
# The discharge_disposition_id codes 11, 13, 14, 19, 20, 21 indicate death or discharge to a non-acute care facility, 
# making 30-day readmission to the *original* hospital impossible/irrelevant.
invalid_discharge_ids = [11, 13, 14, 19, 20, 21]
initial_rows = len(df)
df = df[~df['discharge_disposition_id'].isin(invalid_discharge_ids)]
print(f"Filtered {initial_rows - len(df)} rows due to invalid discharge status.")

In [ ]:
# --- 3. Feature Engineering: Target and Categorical Fields ---

# 3.1. Target Variable: Readmitted_30days
df['Readmitted_30days'] = (df['readmitted'] == '<30').astype(int)
df = df.drop(columns=['readmitted'])
print(f"30-Day Readmission Rate: {df['Readmitted_30days'].mean():.2%}")

In [ ]:
# 3.2. Diagnosis Category (ICD-9 Mapping)
def map_icd9_to_category(icd9_code):
    """Maps a primary ICD-9 code (as a string) to a general category."""
    if pd.isna(icd9_code) or icd9_code in ('?', ''):
        return 'Unknown'
    
    code = str(icd9_code).strip().upper()
    
    if code.startswith('V') or code.startswith('E'):
        return 'External Causes/V-E Codes'
    
    try:
        if '.' in code:
            numeric_prefix = float(code.split('.')[0])
        else:
            numeric_prefix = float(code)

        if 1 <= numeric_prefix <= 139:
            return 'Infectious/Parasitic'
        elif 140 <= numeric_prefix <= 239:
            return 'Neoplasms (Cancer)'
        elif 240 <= numeric_prefix <= 279:
            # Diabetes (250.xx) is here, which is the primary focus
            return 'Endocrine/Metabolic/Immunity'
        elif 280 <= numeric_prefix <= 289:
            return 'Blood/Blood-Forming'
        elif 390 <= numeric_prefix <= 459:
            return 'Circulatory/Cardiovascular'
        elif 460 <= numeric_prefix <= 519:
            return 'Respiratory System'
        elif 520 <= numeric_prefix <= 579:
            return 'Digestive System'
        elif 580 <= numeric_prefix <= 629:
            return 'Genitourinary System'
        elif 800 <= numeric_prefix <= 999:
            return 'Injury/Poisoning/External'
        else:
            return 'Other/Ill-defined'
            
    except ValueError:
        return 'Other/Ill-defined'

# Apply the mapping to the primary diagnosis
df['Diagnosis_Category'] = df['diag_1'].apply(map_icd9_to_category)

In [ ]:
# 3.3. Comorbidity Count Proxy
# This counts how many diagnoses fields (diag_2, diag_3) are *not* missing or 'Other/Ill-defined'
def count_comorbidities(row):
    count = 0
    for diag in ['diag_2', 'diag_3']:
        # If the code is present and not the general 'Other/Ill-defined' category
        if pd.notna(row[diag]) and map_icd9_to_category(row[diag]) not in ['Other/Ill-defined', 'Unknown', 'External Causes/V-E Codes']:
            count += 1
    return count

df['Comorbidity_Count'] = df.apply(count_comorbidities, axis=1)

In [ ]:
3.4. Length of Stay (LOS) Bucket
# Create categories for short, medium, and long stays
bins = [0, 3, 7, np.inf]
labels = ['Short (1-3d)', 'Medium (4-7d)', 'Long (8+d)']
df['LOS_Bucket'] = pd.cut(df['Length_of_Stay'], bins=bins, labels=labels, right=True, include_lowest=True)
df['LOS_Bucket'] = df['LOS_Bucket'].astype(str) # Convert to string for SQL export/modeling

In [ ]:
# 3.5. Medication Change Flag (Medication_Change)
# A change occurred if the 'change' column is 'Ch' or if any specific drug column changed from 'No' to 'Up'/'Down'
# A simplified proxy is using the 'change' column combined with a check on insulin
df['Medication_Change'] = ((df['change'] == 'Ch') | (df['insulin'].isin(['Up', 'Down']))).astype(int)
df = df.drop(columns=['change']) # Drop the intermediate column

In [ ]:
# --- 4. Final Data Selection and Renaming ---

# The original dataset uses numerical IDs for many categorical columns (e.g., admission_type_id).
# We should map these to descriptive strings if possible (a manual step or lookup table needed).
# For now, we will treat them as categorical IDs, which is common practice.

FINAL_COLUMNS = [
    'encounter_id', 'Patient_ID', 'race', 'gender', 'age', 
    'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
    'Diagnosis_Category', 'Length_of_Stay', 'LOS_Bucket', 
    'Num_Inpatient', 'Num_Medications', 'Num_Lab_Procedures', 'Num_Procedures',
    'Comorbidity_Count', 'Medication_Change', 'Readmitted_30days'
]

# Select and harmonize remaining column names
df_cleaned = df[FINAL_COLUMNS].rename(columns={
    'age': 'Age_Group', # We use the existing bracketed age for simplicity
    'Num_Inpatient': 'Num_Prior_Admissions', # Treat this count as prior admissions for modeling purposes
    'encounter_id': 'Encounter_ID',
    'race': 'Race',
    'gender': 'Gender'
})

# Drop diagnosis columns now that we have the category and comorbidity proxy
df_cleaned = df_cleaned.drop(columns=['admission_type_id', 'discharge_disposition_id', 'admission_source_id'])

# Simple imputation for Race and Gender (fill with 'Unknown' category)
df_cleaned['Race'] = df_cleaned['Race'].fillna('Unknown')
df_cleaned['Gender'] = df_cleaned['Gender'].fillna('Unknown')
# Note: Further, deeper imputation or encoding would happen in 03_modeling.

In [ ]:
# --- 5. Save Cleaned Data ---
print(f"Final cleaned shape: {df_cleaned.shape}")

# Ensure the columns match the SQL schema defined previously
# The final CSV should be ready for bulk import!
df_cleaned.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Cleaned data saved to: {PROCESSED_DATA_PATH}")